# 01 - Labelling
1. Generate candidate points (spread over 12 image clusters, independent of Dynamic World / WorldCover).
2. Label them in **QGIS** (free) with a Google Satellite / Esri imagery basemap, following `docs/labelling_protocol.md`.
3. Validate the file and check agreement between labellers.

In [ ]:
# --- Setup (run once per Colab session) ---
!git clone -q https://github.com/veeresh0804/PrithviBodh.git 2>/dev/null || (cd PrithviBodh && git pull -q)
%cd /content/PrithviBodh
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PrithviBodh'      # Earth Engine exports land here
import os; os.makedirs(DRIVE, exist_ok=True)

import sys; sys.path.insert(0, '.')
from prithvibodh.config import load_config
cfg = load_config()
cfg['gee_project'] = 'YOUR_EE_PROJECT_ID'         # <-- your free noncommercial project
from prithvibodh import gee_data as G
G.init(cfg)

In [ ]:
cand = G.candidate_points(cfg)
cand['label'] = ''; cand['labeller'] = ''; cand['confidence'] = ''
cand.to_csv(f'{DRIVE}/candidates.csv', index=False)
print(len(cand), 'candidate points saved to Drive/PrithviBodh/candidates.csv')
cand.cluster.value_counts().sort_index()

**Split the work:** each member labels one third. Give 10% of points to two people (for the agreement check).
Save the finished file as `Drive/PrithviBodh/labels.csv`, and each person's overlap file as `labels_<name>.csv`.

In [ ]:
from prithvibodh.labels import load_labels, agreement, class_counts
lab = load_labels(f'{DRIVE}/labels.csv', cfg)
print(len(lab), 'usable labelled points'); class_counts(lab, cfg)

In [ ]:
# Agreement check on the shared 10% (edit file names)
a = load_labels(f'{DRIVE}/labels_member1.csv', cfg, drop_unsure=False)
b = load_labels(f'{DRIVE}/labels_member2.csv', cfg, drop_unsure=False)
agreement(a, b)      # aim for percent_agreement >= 0.85